# From uncertainty to action: which experiment should we run next?
**Optional AI/ML extension · teams of 2–3 · about 25 minutes**

Imagine that your team is training an ML model. You need to choose its **learning rate**, but every training run takes **two GPU-hours**. You can afford only eight runs.

A very small learning rate may learn too slowly. A very large one may make training unstable. The useful region lies somewhere in between, but we do not know where. Our goal is to finish our runs able to **recommend a good learning rate**, without paying to try every option. That is not quite the same as observing a low number along the way—once training runs are noisy, the two come apart.

We will use a **Gaussian process (GP)** to describe what we know about model performance at tried and untried learning rates. Then we will use that uncertainty to decide which run to buy next.

The ML problem, candidate learning rates, and compute budget will stay fixed. **Your task is to design the Bayesian optimizer:** decide what its GP should assume and how strongly its acquisition rule should explore. You will see how those choices change both the next experiment and the complete search path.


In [ ]:
# Setup only: import libraries and configure paths
from pathlib import Path
import sys
import importlib

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'workshop').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import workshop.gp_optimization as gp_optimization
importlib.reload(gp_optimization)
from workshop.gp_optimization import (
    gp_posterior, lower_confidence_bound, rbf_kernel,
    run_bayesian_optimization,
    suggest_next_index,
)

DATA = ROOT / 'data' / 'public'
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## 1. The casino idea, enlarged

In the casino, one unknown number—win probability $p$—determined the result. Model tuning has an unknown **function**: each learning rate has its own expected validation loss.

| Casino | Model tuning |
|---|---|
| Unknown win probability $p$ | Unknown performance curve $f(x)$ |
| One play | One training run |
| Beta prior over $p$ | GP prior over possible curves |
| Posterior over $p$ | Posterior over curves |
| Decide whether to keep playing | Decide which learning rate to train next |

For a fast and reproducible workshop, a CSV acts as our training service. It holds **simulated** results for 33 possible learning rates, and we reveal a row only when we pay for that run. No real GPU job is launched, and all compute costs in this exercise are illustrative.

The simulation is deliberately built like the real thing, with two properties that stay hidden until Section 5:

- Each learning rate has a **true average loss** $f(x)$ lying on a smooth curve.
- Each training run reports that average **plus noise**: $y = f(x) + \varepsilon$. The number you buy is one draw, not the truth.

The search table contains only those noisy results. A separate reveal table holds the hidden average curve and the simulator's noise scale. It is loaded after the eight decisions, so neither the participants nor the optimization policy can use that information while choosing experiments.


In [ ]:
all_runs = pd.read_csv(DATA / 'gp_tuning_results.csv')
candidate_x = all_runs.log10_learning_rate.to_numpy()
cached_loss = all_runs.validation_loss.to_numpy()

BUDGET = 8
HOURS_PER_RUN = 2
initial_log_rates = [-4.75, -3.50, -1.25]
initial_indices = [
    int(np.flatnonzero(np.isclose(candidate_x, value))[0])
    for value in initial_log_rates
]
observed_indices = initial_indices.copy()

initial_results = all_runs.iloc[observed_indices][
    ['learning_rate', 'log10_learning_rate', 'validation_loss']
].copy()
initial_results.columns = ['Learning rate', 'log10 learning rate', 'Validation loss']
initial_results['Learning rate'] = initial_results['Learning rate'].map(lambda value: f'{value:.2e}')
display(initial_results.reset_index(drop=True))

plt.scatter(candidate_x[observed_indices], cached_loss[observed_indices],
            s=80, color='#2878B5', zorder=3)
plt.xlabel('log10(learning rate)')
plt.ylabel('Validation loss (lower is better)')
plt.title('The three training runs we have paid for', loc='left')
plt.xlim(candidate_x.min(), candidate_x.max())
plt.ylim(.235, .515)
plt.grid(alpha=.15)
plt.show()


**Pause and predict:** where do you think the best learning rate lies? Where are you most uncertain? If you could buy one run now, which value would you try?


## 2. Design your Gaussian process

A Gaussian process is a probability distribution over functions. Before seeing results, it describes many performance curves that we consider plausible. After observing runs, Bayes' rule gives more weight to curves that agree with those results.

With only three observations, we should not expect the GP to determine its assumptions reliably for us. The code cell below is your control panel. Each setting answers a question about the optimization problem, not about the model architecture.

| Control | Units here | What are you telling the optimizer? | Values worth trying |
|---|---|---|---|
| `PRIOR_MEAN` | validation loss | What loss is plausible before nearby evidence exists? | `0.35`, `0.45`, `0.55` |
| `SIGNAL_STD` | validation loss | How much might the hidden average loss vary across learning rates, around the prior mean? | `0.05`, `0.10`, `0.16` |
| `LENGTH_SCALE` | $\log_{10}$(learning rate) | How far does information travel? Smaller allows sharper bends. | `0.10`, `0.50`, `1.50` |
| `NOISE_STD` | validation loss | How much might repeated runs at the same learning rate scatter around its hidden average loss? | `0.005`, `0.015`, `0.040` |
| `EXPLORATION` | none (dimensionless) | How much should uncertainty attract the next experiment? | `0`, `1.25`, `2.50` |

The first four controls define the GP; `EXPLORATION` belongs to the acquisition rule that will use it. A GP prior consists of a mean function and a kernel. In this exercise the mean is the constant `PRIOR_MEAN`, while the radial basis function (RBF) kernel is

$$k(x,x') = \sigma_f^2 \exp\left(-\frac{(x-x')^2}{2\ell^2}\right),$$

where $\sigma_f$ is `SIGNAL_STD` and $\ell$ is `LENGTH_SCALE`. Thus `PRIOR_MEAN` is part of the GP prior, but not part of the kernel. A length scale of `0.50` corresponds to learning rates differing by a factor of $10^{0.50} \approx 3.2$; at that separation their prior correlation is $e^{-1/2} \approx 0.61$. The GP assumes independent Gaussian noise with the same `NOISE_STD` at every rate.

The RBF is a common starting choice when we expect a smooth function in which nearby inputs behave similarly. It is still a modeling assumption, not a consequence of Bayes, and can be a poor choice for abrupt changes, discontinuities, periodic behavior, or variation at several different scales.

There is no universally correct setting. In a real project you would use domain knowledge, repeated runs, historical experiments, and sensitivity checks—not tune these values against hidden answers. If your sampled prior curves include impossible loss values, treat that as useful feedback that the prior is too broad or the objective needs a transformation.

**Your experiment:** change **one control at a time**, predict what will happen, and rerun from this cell through the next-suggestion plot. Then choose one design your team can explain before completing the eight-run search.


In [ ]:
# YOUR OPTIMIZER DESIGN: edit these values, then rerun the cells below.
PRIOR_MEAN = 0.45      # validation-loss units; baseline away from observations
SIGNAL_STD = 0.10      # validation-loss units; vertical curve variation
LENGTH_SCALE = 0.50    # log10-learning-rate units; smaller = sharper bends
NOISE_STD = 0.015      # validation-loss units; run-to-run variation
EXPLORATION = 1.25     # dimensionless; 0 = predicted loss only

gp_settings = dict(
    prior_mean=PRIOR_MEAN,
    signal_std=SIGNAL_STD,
    length_scale=LENGTH_SCALE,
    noise_std=NOISE_STD,
)
x_plot = np.linspace(candidate_x.min(), candidate_x.max(), 500)
Z90 = 1.645

def posterior_at(indices, query, settings=None):
    settings = gp_settings if settings is None else settings
    return gp_posterior(
        candidate_x[indices], cached_loss[indices], query, **settings
    )

def plot_gp(ax, indices, title, next_index=None, settings=None,
            show_acquisition=False):
    active_settings = gp_settings if settings is None else settings
    mean, latent_std, _ = posterior_at(
        indices, x_plot, active_settings
    )
    ax.fill_between(
        x_plot, mean - Z90 * latent_std, mean + Z90 * latent_std,
        color='#2878B5', alpha=.25,
        label='90% band for average loss',
    )
    ax.plot(x_plot, mean, color='#205E86', linewidth=2.5,
            label='Predicted average loss (GP mean)')
    next_y = None
    next_label = 'Next run (predicted average)'
    if show_acquisition:
        candidate_mean, candidate_std, _ = posterior_at(
            indices, candidate_x, active_settings
        )
        decision_score = lower_confidence_bound(
            candidate_mean, candidate_std, EXPLORATION
        )
        visible_score = decision_score.copy()
        visible_score[indices] = np.nan
        ax.plot(candidate_x, visible_score, marker='.', color='#7C3AED',
                linewidth=1.7,
                label='Decision score (mean − uncertainty bonus)')
        if next_index is not None:
            next_y = decision_score[next_index]
            next_label = 'Next run (lowest decision score)'
    ax.scatter(candidate_x[indices], cached_loss[indices], color='#172B4D',
               edgecolor='white', linewidth=.7, s=58, zorder=4,
               label='Observed runs')
    if next_index is not None:
        if next_y is None:
            next_y = posterior_at(
                indices, candidate_x, active_settings
            )[0][next_index]
        ax.scatter(candidate_x[next_index], next_y, marker='*', s=180,
                   color='#D97706', zorder=5,
                   label=next_label)
    ax.set(xlim=(candidate_x.min(), candidate_x.max()),
           xlabel='log10(learning rate)', ylabel='Validation loss', title=title)
    ax.margins(y=.08)
    ax.grid(alpha=.15)

# Draw possible curves from your prior so its assumptions are visible.
prior_x = np.linspace(candidate_x.min(), candidate_x.max(), 180)
prior_covariance = rbf_kernel(
    prior_x, prior_x, LENGTH_SCALE, SIGNAL_STD
)
prior_cholesky = np.linalg.cholesky(
    prior_covariance + np.eye(len(prior_x)) * 1e-10
)
prior_samples = (PRIOR_MEAN +
                 np.random.default_rng(7).normal(size=(5, len(prior_x)))
                 @ prior_cholesky.T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True, sharey=True,
                         layout='constrained')
for sample_number, sample in enumerate(prior_samples):
    axes[0].plot(prior_x, sample, color='#7C3AED', alpha=.62, linewidth=1.7,
                 label='Random draw from GP prior' if sample_number == 0 else None)
axes[0].axhline(PRIOR_MEAN, color='#475569', linestyle='--', linewidth=2,
                label='Prior mean')
axes[0].set(xlim=(candidate_x.min(), candidate_x.max()),
            xlabel='log10(learning rate)', ylabel='Validation loss',
            title='Before data: curves your settings allow')
axes[0].grid(alpha=.15)
plot_gp(axes[1], initial_indices, 'After three runs: updated belief')
legend_items = {}
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        legend_items.setdefault(label, handle)
fig.legend(legend_items.values(), legend_items.keys(),
           loc='outside lower center', ncol=3, frameon=False)
plt.show()


The left panel makes your prior concrete. `PRIOR_MEAN` moves its center, `SIGNAL_STD` changes its vertical spread, and `LENGTH_SCALE` changes how quickly its curves can bend. `NOISE_STD` matters after observations arrive: larger values tell the GP to trust each individual run less.

In the right panel, the blue band describes uncertainty about the average loss function because we have evaluated only three learning rates. It usually narrows where evidence is collected. It is not a range for one future noisy run; `NOISE_STD` still prevents the GP from treating each observed result as exact, which is why the mean curve need not pass through every dot.

Read the band **vertically, one learning rate at a time**: under the GP assumptions, it contains 90% of the posterior probability for the average loss at that learning rate. It is not a simultaneous 90% region for an entire curve.


## 3. Let your optimizer choose the next experiment

We need a repeatable rule for buying the next run. A common acquisition rule combines the GP prediction and its uncertainty into a **decision score** (in validation-loss units):

$$\text{decision score}(x) = \text{predicted average loss}(x) - \kappa \times \text{uncertainty about that average}(x).$$

Lower scores are tried first. A low prediction makes a learning rate attractive; high uncertainty can also lower its score because the true average might be better than our current prediction. Your `EXPLORATION` value is $\kappa$: `0` ignores uncertainty, while larger values give uncertain candidates a bigger bonus. The score guides a decision—it is **not** a prediction of the loss a run will produce.

The mean and uncertainty here describe the latent **average validation loss**, not the noisy result of one future training run. The rule is conventionally called a *lower confidence bound (LCB)*, but here it is a ranking heuristic rather than a guaranteed bound.

This rule answers **which run to buy next**. After the budget is spent, Section 5 uses the updated GP for a different task: recommending the candidate with the lowest posterior mean. Keeping these two rules separate allows an experiment to be valuable for what it teaches, even when it is not the final recommendation.


In [ ]:
candidate_mean, candidate_latent_std, _ = posterior_at(
    initial_indices, candidate_x
)
acquisition = lower_confidence_bound(
    candidate_mean, candidate_latent_std, EXPLORATION
)
next_index = suggest_next_index(
    candidate_mean, candidate_latent_std, initial_indices, EXPLORATION
)
visible_acquisition = acquisition.copy()
visible_acquisition[initial_indices] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), layout='constrained')
plot_gp(axes[0], initial_indices, 'What the GP currently believes', next_index)
axes[1].plot(candidate_x, visible_acquisition, marker='.', color='#7C3AED',
             label='Decision score (mean − uncertainty bonus)')
axes[1].scatter(candidate_x[next_index], acquisition[next_index], marker='*',
                s=180, color='#D97706', zorder=4,
                label='Next run (lowest decision score)')
axes[1].set(xlabel='log10(learning rate)',
            ylabel='Decision score (lower is tried first)',
            title='Where should we train next?')
axes[1].grid(alpha=.15)
legend_items = {}
for ax in axes:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        legend_items.setdefault(label, handle)
fig.legend(legend_items.values(), legend_items.keys(),
           loc='outside lower center', ncol=3, frameon=False)
plt.show()

suggested_rate = all_runs.iloc[next_index].learning_rate
print(f'The GP suggests log10(learning rate) = {candidate_x[next_index]:.3f}')
print(f'That is a learning rate of {suggested_rate:.2e}.')


The two orange stars refer to the same proposed learning rate. Its vertical position on the left is the GP's predicted average loss; on the right it is the optimistic score used for ranking. Compare the suggestion with your intuition. If you change `EXPLORATION` in the control panel and rerun, the GP belief stays the same but this ranking—and possibly the suggestion—changes. If you change a GP setting, both panels can change.

**Pause and experiment:** try at least two optimizer designs while the result is still hidden. Record the first suggestion from each. Then settle on one design your team can justify, rerun from the control panel, and reveal its next result below.


In [ ]:
# Reveal the suggested run. Rerunning resets to the same three starting runs.
observed_indices = [*initial_indices, next_index]

new_run = all_runs.iloc[next_index]
best_before = cached_loss[initial_indices].min()
print(f'New run: learning rate {new_run.learning_rate:.2e}')
print(f'Observed validation loss: {new_run.validation_loss:.4f}')
print(f'Best loss before this run: {best_before:.4f}')

fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
plot_gp(ax, observed_indices, 'Posterior after buying one more run')
ax.scatter(candidate_x[next_index], cached_loss[next_index], marker='*',
           color='#D97706', s=190, zorder=5, label='New result')
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.16), ncol=3, frameon=False)
plt.show()


## 4. Run the optimizer you designed

Now commit to one set of GP and acquisition settings. We will repeat the same cycle: update the GP, score every untried candidate, buy the most attractive run, and update again. The cell below continues **your policy** from the result just revealed until the eight-run budget is exhausted.

The ML setup, candidate learning rates, starting evidence, and budget remain fixed; only the optimizer design is yours. The policy uses only results already revealed at each step and cannot peek at the other cached losses.

In every panel, the dark-blue line is the GP's **predicted average loss** after the runs shown. The purple curve is the **decision score**—predicted average minus the uncertainty bonus—at each untried candidate; it is not another loss prediction. The orange star marks its lowest value, which is the next proposed run. The yellow star marks the newest observed result, so you can see how new evidence changes both curves. After eight runs there is no orange star because the budget is finished.


In [ ]:
selected_indices = run_bayesian_optimization(
    candidate_x, cached_loss, observed_indices, BUDGET,
    **gp_settings, exploration=EXPLORATION,
)
history = all_runs.iloc[selected_indices][
    ['learning_rate', 'log10_learning_rate', 'validation_loss']
].copy().reset_index(drop=True)
history.insert(0, 'Decision', np.arange(1, len(history) + 1))
history.insert(1, 'Chosen by', ['Initial design'] * len(initial_indices) +
               ['GP acquisition'] * (BUDGET - len(initial_indices)))
history['Best loss so far'] = history.validation_loss.cummin()
shown_history = history.copy()
shown_history['learning_rate'] = shown_history.learning_rate.map(lambda value: f'{value:.2e}')
display(shown_history.round({'log10_learning_rate': 3,
                             'validation_loss': 4, 'Best loss so far': 4}))

snapshot_sizes = [3, 4, 6, 8]
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True,
                         layout='constrained')
for ax, count in zip(axes.flat, snapshot_sizes):
    next_choice = selected_indices[count] if count < BUDGET else None
    plot_gp(ax, selected_indices[:count], f'After {count} runs', next_choice,
            show_acquisition=True)
    if count > len(initial_indices):
        newest = selected_indices[count - 1]
        ax.scatter(candidate_x[newest], cached_loss[newest], marker='*',
                   s=190, color='#FACC15', edgecolor='#854D0E',
                   linewidth=1, zorder=6, label='Newest observed result')
legend_items = {}
for ax in axes.flat:
    handles, labels = ax.get_legend_handles_labels()
    for handle, label in zip(handles, labels):
        legend_items.setdefault(label, handle)
fig.legend(legend_items.values(), legend_items.keys(),
           loc='outside lower center', ncol=3, frameon=False)
fig.suptitle('Your settings shape the complete sequence of experiments', fontsize=15)
plt.show()


### Watch every step

The four panels summarize the search. The animation below shows every update, including the fifth and seventh runs omitted from those snapshots. Use its controls to pause or move one frame at a time. At each step, the orange star is the next proposed experiment; in the following frame it becomes the newest observed result, shown in yellow at the validation loss that run actually produced.


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

frame_counts = range(len(initial_indices), BUDGET + 1)
animation_y_values = [cached_loss[selected_indices]]
for count in frame_counts:
    frame_indices = selected_indices[:count]
    frame_mean, frame_std, _ = posterior_at(frame_indices, x_plot)
    animation_y_values.extend([
        frame_mean - Z90 * frame_std,
        frame_mean + Z90 * frame_std,
    ])
    candidate_mean, candidate_std, _ = posterior_at(
        frame_indices, candidate_x
    )
    frame_score = lower_confidence_bound(
        candidate_mean, candidate_std, EXPLORATION
    )
    animation_y_values.append(frame_score)
animation_y_values = np.concatenate(animation_y_values)
animation_span = np.ptp(animation_y_values)
animation_padding = max(.05 * animation_span, .01)
animation_ylim = (
    animation_y_values.min() - animation_padding,
    animation_y_values.max() + animation_padding,
)

animation_fig, animation_ax = plt.subplots(figsize=(11, 5.5))
animation_fig.subplots_adjust(bottom=.28)

def draw_optimization_step(count):
    animation_ax.clear()
    next_choice = selected_indices[count] if count < BUDGET else None
    plot_gp(
        animation_ax, selected_indices[:count],
        f'Optimizer update: {count} of {BUDGET} runs observed',
        next_choice, show_acquisition=True,
    )
    animation_ax.set_ylim(animation_ylim)
    if count > len(initial_indices):
        newest = selected_indices[count - 1]
        animation_ax.scatter(
            candidate_x[newest], cached_loss[newest], marker='*',
            s=190, color='#FACC15', edgecolor='#854D0E',
            linewidth=1, zorder=6, label='Newest observed result',
        )
    animation_ax.legend(
        loc='upper center', bbox_to_anchor=(.5, -.17),
        ncol=3, frameon=False, fontsize=8,
    )

optimization_animation = FuncAnimation(
    animation_fig, draw_optimization_step, frames=frame_counts,
    interval=1400, repeat=True, repeat_delay=1800,
)
plt.close(animation_fig)
HTML(optimization_animation.to_jshtml())


## 5. Final reveal: what did your optimizer do?

Time to reveal what the simulated training service was hiding: the **true average-loss curve** behind the noisy results you bought.

The left panel shows where your optimizer spent its eight-run budget against that true curve. It also contrasts the GP recommendation with simply choosing the run that printed the lowest number. The right panel compares searches that choose their five additional experiments at random; these replays vary the chosen rates but reuse the same cached noisy results.

Before seeing the truth, we fit the GP to all eight results and **recommend the candidate with the lowest posterior mean**. The acquisition rule and recommendation rule have different jobs: acquisition may buy an uncertain experiment to learn from it, while the final recommendation uses the GP's best estimate after all evidence has arrived. In a real project, that recommendation should be confirmed with fresh runs or untouched test data.

**Regret** then measures what that recommendation cost:

$$\text{regret} \;=\; f(\text{the rate you recommended}) \;-\; f(\text{the genuinely best rate})$$

where $f$ is the *true average* loss. Zero regret means no learning rate on the grid would have served you better; larger regret means you left real performance on the table. Regret is not measured on the noisy numbers you observed. In a real project you do not know the true curve, which is exactly why it is useful to practise on a simulation where we can reveal it afterward.


In [ ]:
truth_runs = pd.read_csv(DATA / 'gp_tuning_truth.csv')
if not np.allclose(truth_runs.log10_learning_rate, candidate_x):
    raise ValueError('The reveal table does not match the candidate grid')
true_average = truth_runs.true_average_loss.to_numpy()
true_noise = truth_runs.true_noise_std.to_numpy()
true_best_index = int(np.argmin(true_average))

def recommend_from_gp(indices, settings=None):
    mean = posterior_at(indices, candidate_x, settings)[0]
    return int(np.argmin(mean)), mean

recommended_index, final_mean = recommend_from_gp(selected_indices)
lowest_observed_index = selected_indices[
    int(np.argmin(cached_loss[selected_indices]))
]
search_regret = (true_average[recommended_index] -
                 true_average[true_best_index])
observed_regret = (true_average[lowest_observed_index] -
                   true_average[true_best_index])

rng = np.random.default_rng(42)
available = np.setdiff1d(np.arange(len(candidate_x)), initial_indices)
random_regrets = []
for _ in range(5000):
    extra = rng.choice(available, BUDGET - len(initial_indices), replace=False)
    random_indices = np.r_[initial_indices, extra]
    random_pick, _ = recommend_from_gp(random_indices)
    random_regrets.append(true_average[random_pick] - true_average[true_best_index])
random_regrets = np.asarray(random_regrets)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), layout='constrained')
axes[0].plot(candidate_x, true_average, color='#334155', linewidth=2.2,
             label='TRUE average loss (was hidden)')
points = axes[0].scatter(
    candidate_x[selected_indices], cached_loss[selected_indices],
    c=np.arange(1, BUDGET + 1), cmap='viridis', s=75, edgecolor='white',
    linewidth=.7, zorder=4, label='Your runs (observed values)',
)
axes[0].scatter(candidate_x[lowest_observed_index],
                cached_loss[lowest_observed_index], marker='X', s=120,
                color='#DC2626', edgecolor='white', linewidth=.7, zorder=5,
                label='Lowest observed run')
axes[0].scatter(candidate_x[recommended_index],
                true_average[recommended_index],
                marker='*', s=230, color='#FACC15', edgecolor='#854D0E',
                linewidth=1, zorder=5,
                label='GP recommendation (scored on truth)')
axes[0].set(xlabel='log10(learning rate)', ylabel='Validation loss',
            title='Your budget against the true curve')
axes[0].grid(alpha=.15)
axes[0].legend(frameon=False, fontsize=8, loc='upper right')
colorbar = fig.colorbar(points, ax=axes[0], pad=.02)
colorbar.set_label('Evaluation order')

axes[1].hist(random_regrets, bins=25, color='#94A3B8', edgecolor='white')
axes[1].axvline(search_regret, color='#2878B5', linewidth=2.5,
                label=f'Your regret ({search_regret:.4f})')
axes[1].set(xlabel='True average loss given up by the recommendation (regret)',
            ylabel='Number of random searches',
            title='Same budget, random choices')
axes[1].legend(frameon=False)
axes[1].grid(axis='y', alpha=.15)
plt.show()

matched = np.mean(random_regrets <= search_regret)
print(f'Your GP recommends : learning rate '
      f'{all_runs.iloc[recommended_index].learning_rate:.2e}   '
      f'predicted average {final_mean[recommended_index]:.4f}, '
      f'true average {true_average[recommended_index]:.4f}')
print(f'Lowest observed run: learning rate '
      f'{all_runs.iloc[lowest_observed_index].learning_rate:.2e}   '
      f'true average {true_average[lowest_observed_index]:.4f}, '
      f'regret {observed_regret:.4f}')
print(f'Truly best available: learning rate '
      f'{all_runs.iloc[true_best_index].learning_rate:.2e}   '
      f'true average {true_average[true_best_index]:.4f}')
print(f'Your regret: {search_regret:.4f}   '
      f'(median random-search regret {np.median(random_regrets):.4f})')
print(f'Random search did as well or better in {matched:.1%} of replays.')

print(f'The GP assumed one noise std of {NOISE_STD:.3f}; the simulator used '
      f'{true_noise.min():.3f} to {true_noise.max():.3f} across the grid.')
print(f'\nIllustrative compute used: {BUDGET * HOURS_PER_RUN} GPU-hours')
print(f'Illustrative full-grid compute: {len(all_runs) * HOURS_PER_RUN} GPU-hours')


The optimizer can find a strong configuration without reconstructing every detail of the performance curve. Its uncertainty is useful because it changes **where compute is spent**. Compare your path with another team's: different defensible settings collect different evidence.

The simulator lets us score the recommendation against a truth that would remain unknown in a real project. A low regret on this one replay is useful feedback, but it does not establish that the optimizer's assumptions were appropriate.

Limitations this comparison cannot rule out:

- A smooth GP can miss a narrow, isolated optimum. The optional section below varies the length scale so you can see how much of the search that one assumption controls.
- An acquisition policy can make an unlucky sequence of choices.
- The chosen `NOISE_STD` may not describe the variability of real training runs. Repeated runs or historical experiments are needed to assess it.
- One tuning problem is not evidence about the next one.
- A recommendation selected through repeated validation comparisons can still look optimistic. Confirm it with fresh runs and untouched test data.

Just as sampler diagnostics cannot validate the casino model, one successful optimization replay cannot validate its kernel or noise assumptions.


## 6. Explain the optimizer you designed

The decision you made was the **design of a sequential optimization method**, not a judgment about a particular ML model. Finish these sentences with your team:

- Our prior mean and signal scale represented …
- We chose an exploration value of … because …
- The GP recommended … after eight runs because …
- Before trusting this design on our own project, we would check …

A useful design is not simply the one that happens to win this hidden table. It is one whose assumptions were reasonable **before the reveal**, whose behavior you understand, and whose sensitivity you have checked.

### Take the loop to your own project

This notebook replays cached results only to avoid real GPU cost. In a project, the sequential loop is the same, but the newly proposed point goes to your actual experiment.

Before using the loop, define the candidates, objective, budget, plausible variation, smoothness, and repeatability. Express the objective so lower is better (for example, negate a reward you want to maximize). The small helper here is deliberately transparent and handles one numeric input on a finite grid. For several parameters, categories, constraints, or parallel experiments, use a production Bayesian-optimization implementation—but keep the same habit of stating and stress-testing its assumptions.

The practical lesson is not that every ML problem needs a Gaussian process. It is that representing uncertainty can help an AI/ML system decide **what evidence to collect next**, especially when labels, experiments, API calls, or training runs are expensive.


## Optional: what if our smoothness assumption changes?

The length scale controls how far one observation influences predictions. A short length scale permits rapidly changing curves; a long one assumes a very smooth curve. This controlled comparison tries `0.10`, `0.50`, and `1.50`, then lets each GP complete the full eight-run optimization. On the learning-rate scale, those distances correspond to multiplicative factors of about $1.3$, $3.2$, and $31.6$.

Your other control-panel values stay fixed, as do the starting runs, acquisition rule, candidate results, and budget. Only the length scale changes.

**Pause and predict:** which assumption will leave the most uncertainty between observed learning rates? Will all three searches finish with the same chosen learning rate?


In [ ]:
length_scales = [0.10, 0.50, 1.50]
assumption_names = {0.10: 'Short', 0.50: 'Medium', 1.50: 'Long'}
searches_by_length_scale = {}
comparison_rows = []

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True,
                         layout='constrained')
for ax, length_scale in zip(axes, length_scales):
    alternative = {**gp_settings, 'length_scale': length_scale}
    selected = run_bayesian_optimization(
        candidate_x, cached_loss, initial_indices, BUDGET,
        **alternative, exploration=EXPLORATION,
    )
    searches_by_length_scale[length_scale] = selected
    suggestion = selected[len(initial_indices)]
    recommended_index, recommendation_mean = recommend_from_gp(
        selected, alternative
    )
    comparison_rows.append({
        'Length scale': length_scale,
        'Assumption': assumption_names[length_scale],
        'First GP choice': f'{all_runs.iloc[suggestion].learning_rate:.2e}',
        'Recommended learning rate': f'{all_runs.iloc[recommended_index].learning_rate:.2e}',
        'GP predicted average': recommendation_mean[recommended_index],
        'TRUE average there': true_average[recommended_index],
        'True-average gap (one replay)': (true_average[recommended_index] -
                                          true_average[true_best_index]),
    })
    plot_gp(ax, initial_indices,
            f'{assumption_names[length_scale]} length scale = {length_scale:.2f}'
            f'\nfirst GP choice: x = {candidate_x[suggestion]:.3f}',
            suggestion, alternative)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=4, frameon=False)
fig.suptitle('Same evidence, different beliefs about smoothness', fontsize=15)
plt.show()

comparison = pd.DataFrame(comparison_rows)
display(comparison.round({'Length scale': 2, 'GP predicted average': 4,
                          'TRUE average there': 4,
                          'True-average gap (one replay)': 4}))


The `True-average gap` reports what happened on this one fixed replay. It is **not a leaderboard for choosing a length scale**; that would require many independent optimization tasks or noise realizations. Here, compare how the assumptions change the behavior and evidence collected.

### Follow every step of each optimization

Each sequence begins with the same three observations. The next five panels show the posterior after each GP-guided result arrives. A **yellow star marks the newest result** in that panel; in the last panel it therefore marks the eighth and final evaluation.

Only results available at that step are plotted. The unrevealed cached candidates do not appear in these sequences.


In [ ]:
snapshot_counts = range(len(initial_indices), BUDGET + 1)
for length_scale in length_scales:
    alternative = {**gp_settings, 'length_scale': length_scale}
    selected = searches_by_length_scale[length_scale]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), sharex=True, sharey=True,
                             layout='constrained')
    for ax, count in zip(axes.flat, snapshot_counts):
        title = 'Starting evidence: 3 runs' if count == len(initial_indices) else f'After {count} runs'
        plot_gp(ax, selected[:count], title, settings=alternative)
        if count > len(initial_indices):
            newest = selected[count - 1]
            ax.scatter(candidate_x[newest], cached_loss[newest], marker='*',
                       s=190, color='#FACC15', edgecolor='#854D0E',
                       linewidth=1, zorder=6, label='Newest GP-chosen result')
    legend_items = {}
    for ax in axes.flat:
        handles, labels = ax.get_legend_handles_labels()
        for handle, label in zip(handles, labels):
            legend_items.setdefault(label, handle)
    fig.legend(legend_items.values(), legend_items.keys(),
               loc='outside lower center', ncol=4, frameon=False)
    fig.suptitle(
        f'{assumption_names[length_scale]} smoothness assumption '
        f'(length scale = {length_scale:.2f})', fontsize=15,
    )
    plt.show()


### Retrospective comparison

Section 5 revealed the simulator's true average-loss curve and its true run-to-run noise, so we can now draw both behind each search. None of the three optimizations could see them while choosing.

The gray region is the central 90% range for **one noisy training run** around the true average. It is not uncertainty about the average curve. The star marks the candidate with the **lowest final GP posterior mean** after all eight results.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), sharex=True, sharey=True,
                         layout='constrained')
for ax, length_scale in zip(axes, length_scales):
    alternative = {**gp_settings, 'length_scale': length_scale}
    selected = searches_by_length_scale[length_scale]
    recommended_index, _ = recommend_from_gp(selected, alternative)
    ax.fill_between(
        candidate_x, true_average - Z90 * true_noise,
        true_average + Z90 * true_noise, color='#CBD5E1', alpha=.62,
        label='90% range for one noisy training run', zorder=1,
    )
    ax.plot(candidate_x, true_average, color='#334155', linewidth=2,
            label='TRUE average loss', zorder=3)
    ax.scatter(candidate_x[initial_indices], cached_loss[initial_indices],
               s=64, facecolor='white', edgecolor='#172B4D', linewidth=1.5,
               zorder=4, label='Three starting runs')
    ax.scatter(candidate_x[selected[len(initial_indices):]],
               cached_loss[selected[len(initial_indices):]],
               s=64, color='#2878B5', edgecolor='white', linewidth=.7,
               zorder=4, label='Five GP-chosen runs')
    ax.scatter(candidate_x[recommended_index],
               true_average[recommended_index],
               marker='*', s=220, color='#FACC15', edgecolor='#854D0E',
               linewidth=1, zorder=6, label='GP recommendation')
    ax.set(xlim=(candidate_x.min(), candidate_x.max()), ylim=(.235, .515),
           xlabel='log10(learning rate)', ylabel='Validation loss',
           title=f'{assumption_names[length_scale]} length scale = {length_scale:.2f}'
                 f'\nrecommends x = {candidate_x[recommended_index]:.3f}')
    ax.grid(alpha=.15)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='outside lower center', ncol=5, frameon=False)
fig.suptitle('The smoothness assumption changes the entire search path', fontsize=15)
plt.show()


With the other controls at their defaults, the medium scale (`0.50`) reaches and recommends the best grid point, while the two more extreme assumptions behave differently for reasons we can inspect. That makes the example useful, but it still is not a contest with a reliable winner: the exact gaps come from **one fixed curve and one fixed set of noisy runs**. The gray range puts differences between neighboring average losses in the context of ordinary run-to-run variation.

What we can explain is the mechanism behind each assumption:

- The **short** scale (`0.10`) makes information extremely local. Its prediction returns toward the prior only a small distance from each observation, so five new runs are not enough to connect much of the search space.
- The **medium** scale (`0.50`) shares evidence across a useful neighborhood while still allowing the curve to bend around the good region. On this constructed task, that balance produces a sensible path.
- The **long** scale (`1.50`) treats a wide stretch of learning rates as one gently varying region, so its posterior can look calm and confident while smoothing over a narrow optimum.

This is a sensitivity analysis, not a way to select whichever length scale gets the lowest gap here. It shows why checking only the first suggested point is insufficient: an assumption changes the evidence collected later, and those observations change every subsequent decision. A serious comparison would repeat the exercise across independent noise realizations and representative optimization tasks. In practice, kernel choices should also be justified with domain knowledge and historical experiments.
